# Notebook 1 — Ingestion & Partitioning

**Team 8 — Delhi Air Quality Pipeline**
**Member: [Your Name Here]**

## What this notebook does
This is the first stage of the pipeline. My job is to:
1. Set up PySpark in Colab
2. Load the raw parquet file (`team_8.parquet`)
3. Explore the dataset — find out what's inside it
4. Write an ingestion summary
5. Partition the raw data into year/month folders

The output of this notebook (the `partitioned_data/` folder) is what
Member 2 uses as input for the next stage.


## Step 1: Install PySpark and Java

PySpark runs on the Java Virtual Machine, so we need to install Java first.
Then we install the PySpark Python package.

This takes about 2 minutes the first time.

In [ ]:
# Install dependencies in your VS Code terminal before running:
# pip install pyspark

# Make sure Java 17 is installed on your system.
# You can verify using:
# java -version


In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("Java is set up")

Java is set up


## Step 2: Upload the parquet file

In [5]:
# Place your dataset file in the same folder as this notebook/project.
# Update the filename below if needed.

file_path = "C:\Users\USER\Downloads\DT_project\team_8.parquet"
print(f"Using dataset file: {file_path}")


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (1135515304.py, line 4)

## Step 3: Start a Spark session

A `SparkSession` is the main entry point to PySpark. Think of it as
opening a connection to the Spark engine that will do all the work.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Team8_Ingestion")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Spark version:", spark.version)

## Step 4: Load the parquet file

PySpark reads parquet lazily — it doesn't load everything into memory yet.
It just figures out the structure of the file and waits.

In [ ]:
df = spark.read.parquet("team_8.parquet")
print("Schema of the dataset:")
df.printSchema()

In [ ]:
# count the rows — this is where Spark actually reads the file
row_count = df.count()
print(f"Total rows: {row_count:,}")

In [ ]:
# peek at the first 5 rows
df.show(5, truncate=False)

## Step 5: Ingestion summary

Now I'll print a summary of what's in the dataset so we know
what we're working with.

In [ ]:
print("INGESTION SUMMARY")
print("=" * 60)
print(f"Total records : {df.count():,}")
print(f"Columns       : {len(df.columns)}")

date_range = df.agg(
    F.min("datetime").alias("min_dt"),
    F.max("datetime").alias("max_dt")
).collect()[0]
print(f"Date range    : {date_range['min_dt']} to {date_range['max_dt']}")

print(f"States        : {df.select('state').distinct().count()}")
print(f"Cities        : {df.select('city').distinct().count()}")
print(f"Stations      : {df.select('station_id').distinct().count()}")
print(f"Pollutants    : {df.select('pollutant').distinct().count()}")

In [ ]:
# list all the pollutants
print("Pollutants in the dataset:")
df.select("pollutant").distinct().orderBy("pollutant").show()

In [ ]:
# how many readings per year and month
print("Records per year and month:")
df.groupBy("year", "month").count().orderBy("year", "month").show(30)

In [ ]:
# how many readings per pollutant
print("Records per pollutant:")
df.groupBy("pollutant").count().orderBy(F.desc("count")).show()

In [ ]:
# save the summary as a text file so we have a record
summary_text = f'''INGESTION SUMMARY
{'=' * 60}
Total records : {df.count():,}
Columns       : {len(df.columns)}
Date range    : {date_range['min_dt']} to {date_range['max_dt']}
States        : {df.select('state').distinct().count()}
Cities        : {df.select('city').distinct().count()}
Stations      : {df.select('station_id').distinct().count()}
Pollutants    : {df.select('pollutant').distinct().count()}

Pollutants list:
{[r['pollutant'] for r in df.select('pollutant').distinct().orderBy('pollutant').collect()]}
'''

with open("ingestion_summary.txt", "w") as f:
    f.write(summary_text)
print("Saved ingestion_summary.txt")

## Step 6: Partition the data by year and month

PySpark has a built-in `partitionBy` option for parquet writes.
It automatically creates folders like `year=2024/month=01/`.
This is the standard format used by big data tools (Hive, Spark, Athena)
to organize data on disk.

In [ ]:
(
    df
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet("partitioned_data")
)
print("Partitioning complete!")

In [ ]:
# verify the folder structure was created correctly
!find partitioned_data -type d | sort | head -25

In [ ]:
# check that one partition reads back cleanly
sample_partition = spark.read.parquet("partitioned_data/year=2024/month=01/")
print(f"Rows in Jan 2024 partition: {sample_partition.count():,}")
sample_partition.show(3)

## Step 7: Package and download

Zip the partitioned data and summary so we can hand it off to Member 2.

In [ ]:
!zip -r stage1_output.zip partitioned_data/ ingestion_summary.txt -q
print("Zip created")
files.download("stage1_output.zip")

## What I did (for the class presentation)

> "I handled the ingestion and partitioning layer of our data pipeline. I set up
> PySpark in Colab, loaded the raw 22.5-million-row parquet file, and used Spark's
> `partitionBy` to split the data into Hive-style year/month folders. This is the
> standard pattern used by big data tools — it lets future queries scan only the
> partitions they need instead of the whole file."

## Key concepts to know
- **PySpark** — Python interface to Apache Spark, designed for distributed processing of large datasets
- **Lazy evaluation** — Spark doesn't actually run anything until you ask for a result (like `.count()` or `.show()`)
- **Hive-style partitioning** — the `year=YYYY/month=MM/` folder convention recognized by all major big data tools
- **Parquet** — columnar file format that's fast and compressed
